In [1]:
import pandas as pd
import numpy as np
import torch
import joblib
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                              recall_score, classification_report,
                              confusion_matrix)
import matplotlib.pyplot as plt
import seaborn as sns

c:\Users\Buwaneka Fernando\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

test  = pd.read_csv("../data/final/test.csv")
X_test, y_test = test['input_text'], test['cognitive_label'].values

In [ ]:
# ── Helper: get predictions from a transformer model ─────
def get_transformer_predictions(checkpoint_dir, texts, batch_size=32):
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
    model     = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir)
    model     = model.to(device)
    model.eval()

    all_preds = []
    all_probs = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]
        encoding = tokenizer(
            batch_texts,
            max_length=256,
            padding=True,
            truncation=True,
            return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            outputs = model(**encoding)
            probs   = torch.softmax(outputs.logits, dim=1)
            preds   = torch.argmax(probs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())  # Prob of class 1 (System 1)

    return np.array(all_preds), np.array(all_probs)


# ── Get predictions — all 4 models ────
print("Getting predictions from all models...")

# Baselines
lr_pipeline  = joblib.load("../models/baseline_logreg.pkl")
svm_pipeline = joblib.load("../models/baseline_svm.pkl")

lr_preds   = lr_pipeline.predict(X_test)
svm_preds  = svm_pipeline.predict(X_test)

# Transformers
distil_preds, distil_probs   = get_transformer_predictions(
    "../models/distilbert_checkpoint", X_test.tolist()
)
roberta_preds, roberta_probs = get_transformer_predictions(
    "../models/roberta_checkpoint",    X_test.tolist()
)

print("Predictions collected from all 4 models.")